# Mission 02: Streaming Revenue into Delta Lake

The operations dashboard shows taxi revenue per borough in 15-minute windows. Today a batch job recomputes everything
every hour. Replace it with a **Structured Streaming** job that reads new trip files as they arrive and keeps a
**Delta table** up to date with `MERGE`, survives restarts without double counting, and handles late files.

**Lessons used:** 04 (joins), 08 (Structured Streaming and watermarks), 09 (Delta Lake with Spark).

**Data:** NYC yellow taxi trips, 2024-02-01 to 2024-02-03, delivered as hourly Parquet files; the zone lookup table.

**Services:** none (Spark and Delta run locally in the notebook).

In [ ]:
import shutil
import sys
import tempfile
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from delta import configure_spark_with_delta_pip
from delta.tables import DeltaTable
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

sys.path.insert(0, str(Path.cwd().parent.parent / "tools"))
import datasets                                   # course helper: pinned datasets

warnings.filterwarnings("ignore", message=".*pandas >= 3.0.0.*")
plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False, "figure.dpi": 100})
WORK = Path(tempfile.gettempdir()) / "fzth-m13" / "mission02"
shutil.rmtree(WORK, ignore_errors=True)
INBOX, GOLD, CHECKPOINT = WORK / "inbox", WORK / "gold_revenue", WORK / "checkpoint"
INBOX.mkdir(parents=True)

builder = (SparkSession.builder.master("local[4]").appName("m13-mission02")
           .config("spark.driver.memory", "1500m").config("spark.sql.shuffle.partitions", "4")
           .config("spark.ui.showConsoleProgress", "false").config("spark.sql.session.timeZone", "UTC")
           .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
           .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog"))
spark = configure_spark_with_delta_pip(builder).getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

# The source system: 72 hourly files. Files arrive in three waves; wave 3 also re-delivers two files of hours that
# were missing earlier (late files).
trips = pd.read_parquet(datasets.taxi("2024-02"), columns=["tpep_pickup_datetime", "PULocationID", "total_amount"])
trips = trips[(trips.tpep_pickup_datetime >= "2024-02-01") & (trips.tpep_pickup_datetime < "2024-02-04")]
trips = trips.rename(columns={"tpep_pickup_datetime": "pickup_at", "PULocationID": "pu_zone", "total_amount": "total"})
hours = sorted(trips.pickup_at.dt.floor("h").unique())
late_hours = {hours[10], hours[30]}                 # these two hours are delivered late, in wave 3
waves = {1: [h for h in hours[:24] if h not in late_hours], 2: [h for h in hours[24:48] if h not in late_hours],
         3: hours[48:] + sorted(late_hours)}

def deliver(wave: int) -> int:
    for h in waves[wave]:
        part = trips[trips.pickup_at.dt.floor("h") == h]
        part.to_parquet(INBOX / f"trips_{pd.Timestamp(h):%Y%m%d_%H}.parquet", index=False)
    return len(waves[wave])

print(f"{len(trips):,} trips in {len(hours)} hourly files | late hours: {sorted(str(pd.Timestamp(h)) for h in late_hours)}")

## Tasks

1. **Stream.** Read `INBOX` as a streaming file source (give the schema explicitly; at most 8 files per trigger).
   Join the static zone table to get the borough.
2. **Window.** Count trips and sum revenue per borough in 15-minute event-time windows, with a watermark.
3. **Upsert.** In `foreachBatch`, `MERGE` each micro-batch into the Delta table `GOLD` on (window start, borough),
   so a window that receives more trips later is updated, not duplicated. Use `trigger(availableNow=True)`.
4. **Restart.** Deliver wave 1, run the stream, stop. Deliver wave 2, run again with the same checkpoint. Deliver
   wave 3 (with two late files), run again. Show which files each run processed and that nothing was counted twice.
5. **Check.** Compare `GOLD` with a batch computation over all files. Explain every difference (the watermark).
   Run once more with a watermark long enough to accept the late files and show that the difference disappears.
6. **History.** Use the Delta history and time travel to show the table after each run.
7. **Explain.** Five sentences for the dashboard team: what is exactly-once here, what the watermark costs, and what
   to monitor in production.

In [ ]:
# Tasks 1-7
...

## Submission Checklist

- [ ] The stream uses an explicit schema, a watermark and a checkpoint.
- [ ] Re-running with the same checkpoint processes only new files (shown with counts).
- [ ] The Delta table equals the batch result, except for differences you explain.
- [ ] Delta history shows one version per micro-batch, and time travel works.

A solution is in `02-Streaming-Revenue-into-Delta-solution.ipynb`. Try the mission first.